In [2]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\hp\Desktop\FOOD PRODUCTION PROJECT\data\raw\faostat_crop_data.csv"

df = pd.read_csv(file_path)

In [3]:
df.head()

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,288,Ghana,5312,Area harvested,1701.0,"Beans, dry",1991,1991,ha,NaN,M,Missing value; data cannot exist,NaN
1,QCL,Crops and livestock products,288,Ghana,5510,Production,1701.0,"Beans, dry",1991,1991,t,NaN,M,Missing value; data cannot exist,NaN
2,QCL,Crops and livestock products,288,Ghana,5312,Area harvested,1701.0,"Beans, dry",1992,1992,ha,NaN,M,Missing value; data cannot exist,NaN
3,QCL,Crops and livestock products,288,Ghana,5510,Production,1701.0,"Beans, dry",1992,1992,t,NaN,M,Missing value; data cannot exist,NaN
4,QCL,Crops and livestock products,288,Ghana,5312,Area harvested,1701.0,"Beans, dry",1993,1993,ha,NaN,M,Missing value; data cannot exist,NaN


In [4]:
df.shape


(860, 15)

In [5]:
df['Element'].value_counts()

Element
Area harvested    290
Production        290
Yield             280
Name: count, dtype: int64

In [6]:
df['Flag Description'].value_counts()

Flag Description
Official value                         663
Estimated value                        136
Value imputed by a receiving agency     28
Missing value; data cannot exist        20
Value from external organization        13
Name: count, dtype: int64

df['Item'].value_counts()

In [8]:
sorted(df["Item"].unique())

['Beans, dry', 'Cassava, fresh', 'Maize (corn)', 'Rice', 'Yams']

In [9]:
print(df["Year"].min())
print(df["Year"].max())

1961
2024


In [10]:
df.groupby("Item")["Year"].nunique().sort_values(ascending=False)

Item
Cassava, fresh    64
Rice              64
Maize (corn)      64
Yams              64
Beans, dry        34
Name: Year, dtype: int64

In [11]:
df.groupby("Item")["Flag Description"].value_counts()


Item            Flag Description                   
Beans, dry      Official value                          38
                Estimated value                         20
                Missing value; data cannot exist        20
                Value imputed by a receiving agency     14
Cassava, fresh  Official value                          99
                Estimated value                         87
                Value imputed by a receiving agency      6
Maize (corn)    Official value                         175
                Value from external organization         9
                Estimated value                          7
                Value imputed by a receiving agency      1
Rice            Official value                         178
                Estimated value                         10
                Value from external organization         4
Yams            Official value                         173
                Estimated value                         12
    

In [13]:
df_clean = df[df["Flag"] != "M"].copy()

In [15]:
df_clean.head()

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
20,QCL,Crops and livestock products,288,Ghana,5312,Area harvested,1701.0,"Beans, dry",2001,2001,ha,101500.0,A,Official value,NaN
21,QCL,Crops and livestock products,288,Ghana,5412,Yield,1701.0,"Beans, dry",2001,2001,kg/ha,615.8,A,Official value,NaN
22,QCL,Crops and livestock products,288,Ghana,5510,Production,1701.0,"Beans, dry",2001,2001,t,62500.0,A,Official value,NaN
23,QCL,Crops and livestock products,288,Ghana,5312,Area harvested,1701.0,"Beans, dry",2002,2002,ha,180600.0,A,Official value,NaN
24,QCL,Crops and livestock products,288,Ghana,5412,Yield,1701.0,"Beans, dry",2002,2002,kg/ha,775.2,A,Official value,NaN


In [16]:
selected_crops = [
    "Maize (corn)",
    "Rice",
    "Cassava, fresh",
    "Yams"
]

df_clean = df_clean[df_clean["Item"].isin(selected_crops)].copy()

In [17]:
crop_data = df_clean.pivot_table(
    index=["Area", "Item", "Year"],
    columns="Element",
    values="Value",
    aggfunc="first"
).reset_index()

In [18]:
crop_data.head()

Element,Area,Item,Year,Area harvested,Production,Yield
0,Ghana,"Cassava, fresh",1961,140000.0,1050000.0,7500.0
1,Ghana,"Cassava, fresh",1962,140000.0,1050000.0,7500.0
2,Ghana,"Cassava, fresh",1963,158554.0,1194265.0,7532.2
3,Ghana,"Cassava, fresh",1964,162803.0,1228608.0,7546.6
4,Ghana,"Cassava, fresh",1965,150000.0,1400000.0,9333.3


In [20]:
# Standardize crop names

df_clean["Item"] = df_clean["Item"].replace({
    "Maize (corn)": "Maize",
    "Cassava, fresh": "Cassava"
})

In [21]:
df_clean["Item"].unique()

array(['Cassava', 'Maize', 'Rice', 'Yams'], dtype=object)

In [23]:
df_clean = df_clean[
    [
        "Area",
        "Item",
        "Element",
        "Year",
        "Unit",
        "Value",
        "Flag",
        "Flag Description",
    ]
]

In [24]:
crop_data = df_clean.pivot_table(
    index=["Area", "Item", "Year"],
    columns="Element",
    values="Value",
    aggfunc="first"
).reset_index()

In [25]:
crop_data.columns.name = None

crop_data = crop_data.rename(columns={
    "Area harvested": "Area_Harvested_ha",
    "Production": "Production_tonnes",
    "Yield": "Yield_kg_per_ha"
})

In [26]:
crop_data.head()

,Area,Item,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha
0,Ghana,Cassava,1961,140000.0,1050000.0,7500.0
1,Ghana,Cassava,1962,140000.0,1050000.0,7500.0
2,Ghana,Cassava,1963,158554.0,1194265.0,7532.2
3,Ghana,Cassava,1964,162803.0,1228608.0,7546.6
4,Ghana,Cassava,1965,150000.0,1400000.0,9333.3


In [27]:
crop_data.isna().sum()

Area                 0
Item                 0
Year                 0
Area_Harvested_ha    0
Production_tonnes    0
Yield_kg_per_ha      0
dtype: int64

In [28]:
crop_data = crop_data.sort_values(
    by=["Year", "Item"]
).reset_index(drop=True)

In [29]:
crop_data = crop_data.rename(columns={
    "Item": "crop",
    "Year": "year",
    "Area Harvested": "area_harvested_ha",
    "Production": "production_tonnes",
    "Yield": "yield_kg_per_ha"
})

In [30]:
crop_data.head()

,Area,crop,year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha
0,Ghana,Cassava,1961,140000.0,1050000.0,7500.0
1,Ghana,Maize,1961,237000.0,226000.0,953.6
2,Ghana,Rice,1961,27518.0,30400.0,1104.7
3,Ghana,Yams,1961,150000.0,1100000.0,7333.3
4,Ghana,Cassava,1962,140000.0,1050000.0,7500.0


In [31]:
crop_data.to_csv(
    "../data/processed/crop_production_clean.csv",
    index=False
)